# 16 Building a Multi-Agent Subgraph Workflow

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Construye un **flujo multi-agente basado en subgrafos**. Define un subgrafo con dos
nodos-agente, `researcher` (investiga, etiqueta sus mensajes como `sub_researcher`) y
`summarizer` (resume, `sub_summarizer`), encadenados `researcher → summarizer`.
`build_subgraph(llm)` compila ese subgrafo como una unidad invocable desde el flujo
principal.

El notebook une, en orden de dependencia, `llm_provider` (`ChatAnthropic`),
`parent_graph` (define y compila el subgrafo), `subgraph` (variante equivalente) y `main`
(invoca el grafo y muestra la traza).

## Ejemplo de uso

**Datos de interacción que espera el agente.** Subgrafo `researcher → summarizer` sin pausa;
concluye en un `invoke`.

- Entrada inicial esperada: `{"messages": [HumanMessage(content="<tarea>")]}`.
- La salida acumula los mensajes de los sub-agentes (`sub_researcher`, `sub_summarizer`).

```python
from langchain_core.messages import HumanMessage

llm = get_llm()
app = parent_graph.build_subgraph(llm)
final_state = app.invoke(
    {"messages": [HumanMessage(content="Resume las ventajas de LangGraph")]}
)                                                   # researcher→summarizer
for i, m in enumerate(final_state["messages"], 1):
    name = getattr(m, "name", None) or "user/system"
    print(f"{i:02d}. [{name.upper()}] {m.content}\n")
```

In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

In [2]:

# Load environment variables from .env
load_dotenv()

True

In [3]:


def get_llm():
    # Read API key and model name from environment
    api_key = os.getenv("ANTHROPIC_API_KEY")
    model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

    # Fail fast if API key is missing
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing")

    # Return configured Claude LLM
    return ChatAnthropic(
        model=model,
        api_key=api_key,
        temperature=0,  # Deterministic output
    )

In [4]:
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END

In [5]:


class SubgraphState(MessagesState):
    summary: str

In [6]:


def researcher_node(llm):
    def node(state: SubgraphState) -> dict:
        # Visual indicator during execution
        print("   ↳ Subgraph: Researcher working...")

        # Extract the latest user-provided topic
        topic = state["messages"][-1].content

        # System instruction defining the Researcher's role
        sys = SystemMessage(
            content="Role: Researcher\nProvide key background points."
        )

        # Invoke the LLM with role + topic
        resp = llm.invoke([sys, HumanMessage(content=topic)])
        summary = resp.content.strip()

        # Store result in shared state and message history
        return {
            "summary": summary,  # Passed to the next node
            "messages": [
                HumanMessage(
                    content=summary,
                    name="sub_researcher"
                )
            ],
        }

    return node

In [7]:


def summarizer_node(llm):
    def node(state: SubgraphState) -> dict:
        # Visual indicator during execution
        print("   ↳ Subgraph: Summarizer refining output...")

        # System instruction defining the Summarizer's role
        sys = SystemMessage(
            content="Role: Summarizer\nCreate a concise, polished summary."
        )

        # Use the researcher's summary as input
        resp = llm.invoke(
            [sys, HumanMessage(content=state["summary"])]
        )

        # Append final refined output to message history
        return {
            "messages": [
                HumanMessage(
                    content=resp.content.strip(),
                    name="sub_summarizer",
                )
            ]
        }

    return node

In [8]:


def build_subgraph(llm):
    graph = StateGraph(SubgraphState)

    # Register subgraph nodes
    graph.add_node("researcher", researcher_node(llm))
    graph.add_node("summarizer", summarizer_node(llm))

    # Define linear execution flow
    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "summarizer")
    graph.add_edge("summarizer", END)

    # Compile into an executable subgraph
    return graph.compile()

In [9]:
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END

In [10]:


class SubgraphState(MessagesState):
    summary: str

In [11]:


def researcher_node(llm):
    def node(state: SubgraphState) -> dict:
        # Visual indicator during execution
        print("   ↳ Subgraph: Researcher working...")

        # Extract the latest user-provided topic
        topic = state["messages"][-1].content

        # System instruction defining the Researcher's role
        sys = SystemMessage(
            content="Role: Researcher\nProvide key background points."
        )

        # Invoke the LLM with role + topic
        resp = llm.invoke([sys, HumanMessage(content=topic)])
        summary = resp.content.strip()

        # Store result in shared state and message history
        return {
            "summary": summary,  # Passed to the next node
            "messages": [
                HumanMessage(
                    content=summary,
                    name="sub_researcher"
                )
            ],
        }

    return node

In [12]:


def summarizer_node(llm):
    def node(state: SubgraphState) -> dict:
        # Visual indicator during execution
        print("   ↳ Subgraph: Summarizer refining output...")

        # System instruction defining the Summarizer's role
        sys = SystemMessage(
            content="Role: Summarizer\nCreate a concise, polished summary."
        )

        # Use the researcher's summary as input
        resp = llm.invoke(
            [sys, HumanMessage(content=state["summary"])]
        )

        # Append final refined output to message history
        return {
            "messages": [
                HumanMessage(
                    content=resp.content.strip(),
                    name="sub_summarizer",
                )
            ]
        }

    return node

In [13]:


def build_subgraph(llm):
    graph = StateGraph(SubgraphState)

    # Register subgraph nodes
    graph.add_node("researcher", researcher_node(llm))
    graph.add_node("summarizer", summarizer_node(llm))

    # Define linear execution flow
    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "summarizer")
    graph.add_edge("summarizer", END)

    # Compile into an executable subgraph
    return graph.compile()

In [14]:
from langchain_core.messages import HumanMessage


In [15]:


def print_trace(messages):
    print("\n--- Execution Trace ---\n")

    for i, m in enumerate(messages, start=1):
        name = getattr(m, "name", None)
        role = name.upper() if name else "USER/SYSTEM"

        print(f"{i:02d}. [{role}]")
        print(m.content)
        print()

In [16]:


def main():
    # Initialize shared LLM instance
    llm = get_llm()

    # Build graph using build_subgraph (definido en celdas anteriores del notebook)
    app = build_subgraph(llm)

    # Accept user input
    task = input("Enter a task (or exit()): ").strip()

    if task.lower() in {"exit()", "exit", "quit"}:
        print("\nExiting.\n")
        return

    print("\n--- Workflow Started ---\n")

    # Invoke the graph
    final_state = app.invoke(
        {"messages": [HumanMessage(content=task)]}
    )

    print_trace(final_state["messages"])
    print("-" * 70 + "\n")

In [18]:


if __name__ == "__main__":
    main()


--- Workflow Started ---

   ↳ Subgraph: Researcher working...
   ↳ Subgraph: Summarizer refining output...

--- Execution Trace ---

01. [USER/SYSTEM]
Calcular el fibonacci de 20

02. [SUB_RESEARCHER]
# Fibonacci de 20

## Método de cálculo

La secuencia de Fibonacci se define como:
- F(0) = 0
- F(1) = 1
- F(n) = F(n-1) + F(n-2)

## Secuencia hasta F(20)

| n | F(n) |
|---|------|
| 0 | 0 |
| 1 | 1 |
| 2 | 1 |
| 3 | 2 |
| 4 | 3 |
| 5 | 5 |
| 6 | 8 |
| 7 | 13 |
| 8 | 21 |
| 9 | 34 |
| 10 | 55 |
| 11 | 89 |
| 12 | 144 |
| 13 | 233 |
| 14 | 377 |
| 15 | 610 |
| 16 | 987 |
| 17 | 1,597 |
| 18 | 2,584 |
| 19 | 4,181 |
| **20** | **6,765** |

## ✅ Resultado

$$F(20) = \mathbf{6{,}765}$$

> También puede calcularse con la **fórmula de Binet**:
> $$F(n) = \frac{\phi^n - \psi^n}{\sqrt{5}}, \quad \phi = \frac{1+\sqrt{5}}{2} \approx 1.618$$

03. [SUB_SUMMARIZER]
# Resumen: Fibonacci de 20

La secuencia de Fibonacci parte de F(0) = 0 y F(1) = 1, donde cada término es la suma de los dos anteriore